# Security ML Lab — Exploratory Analysis
Connect to Elasticsearch, pull recent events, and explore anomaly scores.

In [5]:
from elasticsearch import Elasticsearch
import pandas as pd

es = Elasticsearch("http://elasticsearch:9200")

resp = es.search(
    index="security-scores-if",
    body={
          "query": {"term": {"ml.is_anomaly": True}},
          "sort": [{"ml.anomaly_score": "desc"}],
          "size": 10,
          "_source": [
              "ml.anomaly_score", "ml.llm_triage",
              "process.name", "process.parent.name",
              "event.category", "user.name", "host.name",
              "source_dataset"
          ]
      }
  )

rows = []
for h in resp["hits"]["hits"]:
      s = h["_source"]
      triage = (s.get("ml") or {}).get("llm_triage", {})
      rows.append({
          "score":     s["ml"]["anomaly_score"],
          "process":   (s.get("process") or {}).get("name"),
          "category":  (s.get("event") or {}).get("category"),
          "dataset":   s.get("source_dataset"),
          "technique": triage.get("attack_technique"),
          "fp":        triage.get("fp_assessment"),
      })

df = pd.DataFrame(rows)
df

,score,process,category,dataset,technique,fp
0,1.000000,cmd.exe,process_creation,empire_psexec_lateral,T1059.001,low
1,0.948693,powershell.exe,process_creation,empire_wmi_lateral,T1059.001,low
2,0.942527,powershell.exe,process_creation,empire_psexec_lateral,T1059.001,low
3,0.915881,cmd.exe,process_creation,empire_psexec_lateral,T1059.001,low
4,0.913529,officec2rclient.exe,process_creation,empire_psexec_lateral,T1059.001,low
5,0.908171,usoclient.exe,process_creation,empire_wmi_lateral,T1059.001,low
6,0.904733,conhost.exe,process_creation,empire_psexec_lateral,T1059.001,low
7,0.895412,wmiprvse.exe,process_creation,empire_launcher_exec,T1059.001,low
8,0.889718,powershell.exe,process_creation,empire_wmi_lateral,T1059.001,low
9,0.889123,whoami.exe,process_creation,empire_psexec_lateral,T1059.001,low


In [4]:
from elasticsearch import Elasticsearch
import pandas as pd
import plotly.express as px

# Use 'elasticsearch' when running inside Docker, 'localhost' otherwise
client = Elasticsearch('http://localhost:9200')
print(client.info()['version']['number'])

ModuleNotFoundError: No module named 'plotly'

In [ ]:
resp = client.search(
    index='siem-logs',
    query={'match_all': {}},
    size=500,
    sort=[{'timestamp': 'desc'}],
)
df = pd.DataFrame([h['_source'] for h in resp['hits']['hits']])
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

In [ ]:
# Score distribution
df['anomaly_score'] = pd.to_numeric(df['anomaly_score'], errors='coerce')
px.histogram(df, x='anomaly_score', nbins=50, title='Anomaly Score Distribution')

In [ ]:
# Top anomalous source IPs
df[df['anomaly_score'] > 0.75].groupby('source_ip')['anomaly_score'].mean().sort_values(ascending=False).head(20)

In [ ]:
# Run scoring directly from notebook
import sys; sys.path.insert(0, '..')
from src.models.anomaly import run
run(hours=24, method='isolation_forest')